# Notebook vấn đáp đồ án 19

**Đề tài:** Xây dựng hệ thống phát hiện và lọc spam email  
**Mục tiêu notebook:** chạy lại toàn bộ pipeline của đồ án từ dữ liệu gốc đến kết quả cuối cùng, theo đúng thứ tự để có thể báo cáo từng cell khi vấn đáp.

Luồng thực hiện trong notebook:

1. Kiểm tra dữ liệu gốc
2. Nạp dữ liệu bằng đúng pipeline của project
3. Parse email và tiền xử lý văn bản
4. Xem cấu hình TF-IDF
5. Chia train/test
6. Biểu diễn email bằng vector TF-IDF
7. Chạy baseline
8. Huấn luyện các mô hình chính
9. Đánh giá chi tiết mô hình SVM dùng để demo
10. Phân tích lỗi và giải thích token
11. Dự đoán thử một số câu mẫu

**Lưu ý:** Notebook này không chạy Streamlit. Khi cần demo giao diện, mở terminal riêng và chạy:

```bash
streamlit run app.py
```


## Colab Setup

Notebook này đã được chỉnh để chạy trên **Google Colab**.

Cell setup dưới đây sẽ:
1. clone repo từ GitHub nếu đang chạy trên Colab,
2. cài các thư viện trong `requirements.txt`,
3. chuyển vào đúng thư mục project trước khi chạy các cell còn lại.

Chỉ cần chạy cell setup này một lần đầu tiên, sau đó chạy tiếp từ trên xuống dưới.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/xuanduong1009/DASpamMail.git"
PROJECT_NAME = "DASpamMail"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    project_root = Path("/content") / PROJECT_NAME
    if not project_root.exists():
        print("Cloning project from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)
    os.chdir(project_root)
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    project_root = Path.cwd()
    if project_root.name == "notebooks":
        project_root = project_root.parent
    os.chdir(project_root)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("Current working directory:", Path.cwd())
print("Running in Colab:", IN_COLAB)
print("Dataset exists:", (Path.cwd() / "data" / "bilingual").exists())


## 0. Chuẩn bị môi trường chạy notebook

Cell dưới đây chỉ thiết lập đường dẫn project, import thư viện và bật chế độ hiển thị bảng dễ nhìn hơn.


In [2]:
from pathlib import Path
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore")

print("Project root:", ROOT)


Project root: c:\Users\ADMIN\Downloads\DASpamMail-main


## 0.1 Import các module của đồ án

Ở bước này, notebook không viết lại logic riêng mà dùng trực tiếp các hàm trong source code của project.  
Nhờ vậy, kết quả trong notebook sẽ bám sát kết quả thật của đồ án.


In [3]:
import sklearn

import src.models.train as train_module
from src.config import DATASET_DIR, MODELS_DIR, RESULTS_DIR, VECTORIZER_PARAMS
from src.features.vectorize import build_stopwords, build_vectorizer
from src.models.evaluate import (
    build_classification_report,
    build_error_analysis,
    compute_metrics,
    get_scores,
)
from src.utils.dataset import load_emails
from src.utils.email_parse import parse_email_bytes
from src.utils.io import ensure_dir, read_bytes
from src.utils.text_clean import clean_text

print("scikit-learn version:", sklearn.__version__)
print("Dataset dir:", DATASET_DIR)
print("Models dir:", MODELS_DIR)
print("Results dir:", RESULTS_DIR)


scikit-learn version: 1.6.1
Dataset dir: C:\Users\ADMIN\Downloads\DASpamMail-main\data\bilingual
Models dir: C:\Users\ADMIN\Downloads\DASpamMail-main\models
Results dir: C:\Users\ADMIN\Downloads\DASpamMail-main\reports\results


## 1. Kiểm tra dữ liệu gốc trong thư mục nguồn

Phần này cho thấy đồ án xuất phát từ thư mục email thật gồm hai nhãn:
- `ham`: email hợp lệ
- `spam`: email rác


In [4]:
ham_files = sorted([p for p in (DATASET_DIR / "ham").rglob("*") if p.is_file()])
spam_files = sorted([p for p in (DATASET_DIR / "spam").rglob("*") if p.is_file()])

raw_count_df = pd.DataFrame(
    [
        {"label": "ham", "raw_file_count": len(ham_files)},
        {"label": "spam", "raw_file_count": len(spam_files)},
        {"label": "total", "raw_file_count": len(ham_files) + len(spam_files)},
    ]
)
raw_count_df


,label,raw_file_count
0,ham,3957
1,spam,1750
2,total,5707


## 2. Nạp dữ liệu bằng đúng pipeline của đồ án

Cell này dùng lại `load_emails()` để đọc toàn bộ email từ thư mục nguồn.  
Ta đồng thời so sánh hai trường hợp:
- chưa loại trùng (`dedup=False`)
- có loại trùng theo nội dung (`dedup=True`)


In [5]:
emails_df = load_emails(DATASET_DIR, dedup=False)
emails_df_dedup = load_emails(DATASET_DIR, dedup=True)

summary_df = pd.DataFrame(
    [
        {
            "dataset_view": "loaded_without_dedup",
            "rows": len(emails_df),
            "ham": int((emails_df["label"] == 0).sum()),
            "spam": int((emails_df["label"] == 1).sum()),
        },
        {
            "dataset_view": "loaded_with_dedup",
            "rows": len(emails_df_dedup),
            "ham": int((emails_df_dedup["label"] == 0).sum()),
            "spam": int((emails_df_dedup["label"] == 1).sum()),
        },
    ]
)
summary_df


load spam: 100%|██████████| 1750/1750 [00:00<00:00, 4885.80it/s]


,dataset_view,rows,ham,spam
0,loaded_without_dedup,5691,3957,1734
1,loaded_with_dedup,5288,3637,1651


In [6]:
emails_df.head(5)[["label_name", "path", "text"]]


,label_name,path,text
0,ham,C:\Users\ADMIN\Downloads\DASpamMail-main\data\bilingual\ham\0001.1999-12-10.farmer.ham.txt,christmas tree farm pictures
1,ham,C:\Users\ADMIN\Downloads\DASpamMail-main\data\bilingual\ham\0002.1999-12-13.farmer.ham.txt,"vastar resources , inc .\ngary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 ..."
2,ham,C:\Users\ADMIN\Downloads\DASpamMail-main\data\bilingual\ham\0003.1999-12-14.farmer.ham.txt,calpine daily gas nomination\n- calpine daily gas nomination 1 . doc
3,ham,C:\Users\ADMIN\Downloads\DASpamMail-main\data\bilingual\ham\0004.1999-12-14.farmer.ham.txt,re : issue\nfyi - see note below - already done .\nstella\n- - - - - - - - - - - - - - - - - - - - - - forwarded by ...
4,ham,C:\Users\ADMIN\Downloads\DASpamMail-main\data\bilingual\ham\0005.1999-12-14.farmer.ham.txt,meter 7268 nov allocation\nfyi .\n- - - - - - - - - - - - - - - - - - - - - - forwarded by lauri a allen / hou / ect...


## 3. Minh họa quá trình parse email và làm sạch văn bản

Đây là phần rất quan trọng khi trình bày, vì nó cho thấy email đi từ raw bytes sang subject/body, rồi sang văn bản đã được chuẩn hóa.


In [7]:
def preview_email(path: Path) -> dict:
    raw = read_bytes(path)
    subject, body = parse_email_bytes(raw)
    combined = f"{subject}\n{body}".strip()
    cleaned = clean_text(combined)
    return {
        "file": path.name,
        "subject": subject[:120],
        "raw_preview": combined[:240],
        "clean_preview": cleaned[:240],
    }


preview_df = pd.DataFrame(
    [
        {"label": "ham", **preview_email(ham_files[0])},
        {"label": "spam", **preview_email(spam_files[0])},
    ]
)
preview_df


,label,file,subject,raw_preview,clean_preview
0,ham,0001.1999-12-10.farmer.ham.txt,christmas tree farm pictures,christmas tree farm pictures,christmas tree farm pictures
1,spam,0006.2003-12-18.GP.spam.txt,dobmeos with hgh my energy level has gone up ! stukm,dobmeos with hgh my energy level has gone up ! stukm\nintroducing\ndoctor - formulated\nhgh\nhuman growth hormone - ...,dobmeos with hgh my energy level has gone up ! stukm introducing doctor - formulated hgh human growth hormone - also...


## 4. Stopwords và cấu hình TF-IDF

Phần này gắn trực tiếp với môn Truy hồi thông tin.  
Nhóm biểu diễn mỗi email thành vector TF-IDF, có dùng stopwords Anh - Việt để giảm nhiễu.


In [8]:
stopwords = build_stopwords()

print("Vectorizer params:", VECTORIZER_PARAMS)
print("Số lượng stopwords:", len(stopwords) if stopwords else 0)
print("20 stopwords đầu tiên:")
print(stopwords[:20] if stopwords else [])


Vectorizer params: {'ngram_range': (1, 2), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True, 'norm': 'l2'}
Số lượng stopwords: 372
20 stopwords đầu tiên:
['a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'ai', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 'amongst']


## 5. Chia train/test theo đúng pipeline của đồ án

Ở đây ta dùng lại `prepare_data()` để đảm bảo notebook chia tập giống hệt code train chính thức:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`


In [9]:
df_all, train_df, test_df, X_train, X_test, y_train, y_test = train_module.prepare_data(
    DATASET_DIR,
    dedup=False,
    return_frames=True,
)

split_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "ham": int((train_df["label"] == 0).sum()),
            "spam": int((train_df["label"] == 1).sum()),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "ham": int((test_df["label"] == 0).sum()),
            "spam": int((test_df["label"] == 1).sum()),
        },
    ]
)
split_df


load spam: 100%|██████████| 1750/1750 [00:00<00:00, 3980.45it/s]


,split,rows,ham,spam
0,train,4552,3165,1387
1,test,1139,792,347


## 6. Fit TF-IDF trên tập train

Cell này chưa huấn luyện classifier mà chỉ kiểm tra đặc trưng TF-IDF:
- số lượng email train
- số chiều đặc trưng
- độ thưa của ma trận
- token phổ biến và token hiếm


In [10]:
vectorizer = build_vectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)

vocab_size = len(vectorizer.get_feature_names_out())
density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])

print("TF-IDF matrix shape:", X_train_tfidf.shape)
print("Vocabulary size:", vocab_size)
print("Matrix density:", round(density, 6))

idf_df = pd.DataFrame(
    {
        "token": vectorizer.get_feature_names_out(),
        "idf": vectorizer.idf_,
    }
)

print("\nToken có IDF thấp nhất (rất phổ biến):")
display(idf_df.sort_values("idf").head(10))

print("\nToken có IDF cao nhất (rất hiếm):")
display(idf_df.sort_values("idf", ascending=False).head(10))


TF-IDF matrix shape: (4552, 56609)
Vocabulary size: 56609
Matrix density: 0.001656

Token có IDF thấp nhất (rất phổ biến):


,token,idf
33902,numtoken,1.176584
35134,numtoken numtoken,1.434659
49022,subject,2.316935
16362,enron,2.340993
50938,thanks,2.349425
44611,s,2.356222
6715,cc,2.459406
20127,gas,2.615607
26573,know,2.617819
23015,hpl,2.659657



Token có IDF cao nhất (rất hiếm):


,token,idf
56546,đ nhấn,8.324929
56545,đ,8.324929
31,_ body,8.324929
28,_ alternative,8.324929
24,99000vnđ thẻ,8.324929
22,500000vnđ sử,8.324929
21,500000vnđ duyệt,8.324929
16,200000vnd sử,8.324929
56541,án đánh,8.324929
56531,zwiers jeff,8.324929


## 7. Chạy các baseline đơn giản

Mục đích của baseline là chứng minh rằng cách làm học máy có hiệu quả cao hơn hẳn cách đoán đơn giản:
- `majority`: luôn đoán theo lớp chiếm đa số
- `keyword`: đoán spam nếu xuất hiện từ khóa spam thủ công


In [11]:
baseline_df = pd.DataFrame(train_module.run_baselines(X_train, y_train, X_test, y_test))
baseline_df = baseline_df.sort_values(["f1_spam", "accuracy"], ascending=[False, False]).reset_index(drop=True)
baseline_df[
    ["model", "precision_spam", "recall_spam", "f1_spam", "accuracy", "fp", "fn"]
]


,model,precision_spam,recall_spam,f1_spam,accuracy,fp,fn
0,keyword,0.589226,0.504323,0.543478,0.741879,122,172
1,majority,0.000000,0.000000,0.000000,0.695347,0,347


## 8. Huấn luyện các mô hình chính

Đây là cell chạy nặng nhất trong notebook.  
Nó sẽ huấn luyện lần lượt:
- Naive Bayes
- Linear SVM
- Random Forest
- Logistic Regression
- XGBoost nếu môi trường có cài

Kết quả sẽ được gom lại thành bảng benchmark để so sánh.


In [12]:
trained_models = {}
model_results = []

model_names = ["nb", "svm", "rf", "lr"]
if train_module.XGBClassifier is not None:
    model_names.append("xgb")

for model_name in model_names:
    print(f"Training model: {model_name}")
    model, metrics = train_module.train_model(
        model_name=model_name,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )
    trained_models[model_name] = model
    model_results.append(metrics)

results_df = pd.DataFrame(baseline_df.to_dict("records") + model_results)
results_df = results_df.sort_values(["f1_spam", "accuracy"], ascending=[False, False]).reset_index(drop=True)

results_df[
    [
        "model",
        "precision_spam",
        "recall_spam",
        "f1_spam",
        "accuracy",
        "fp",
        "fn",
        "macro_f1",
        "balanced_accuracy",
        "roc_auc",
    ]
]


Training model: nb
Training model: svm
Training model: rf
Training model: lr


,model,precision_spam,recall_spam,f1_spam,accuracy,fp,fn,macro_f1,balanced_accuracy,roc_auc
0,lr,0.979827,0.979827,0.979827,0.987709,7,7,0.985494,0.985494,0.996532
1,svm,0.976945,0.976945,0.976945,0.985953,8,8,0.983422,0.983422,0.997919
2,nb,0.966102,0.985591,0.975749,0.985075,12,5,0.982484,0.985220,0.996860
3,rf,0.960227,0.974063,0.967096,0.979807,14,9,0.976265,0.978193,0.997981
4,keyword,0.589226,0.504323,0.543478,0.741879,122,172,0.681776,0.675141,NaN
5,majority,0.000000,0.000000,0.000000,0.695347,0,347,0.410150,0.500000,NaN


## 9. Lưu kết quả riêng của notebook

Để tránh ghi đè file chuẩn của project, notebook lưu kết quả sang các file riêng có tiền tố `notebook_`.


In [ ]:
ensure_dir(MODELS_DIR)
ensure_dir(RESULTS_DIR)

notebook_metrics_path = RESULTS_DIR / "notebook_metrics.csv"
notebook_eval_path = RESULTS_DIR / "notebook_eval_metrics.csv"
notebook_cls_path = RESULTS_DIR / "notebook_classification_report.csv"
notebook_err_path = RESULTS_DIR / "notebook_misclassified_examples.csv"

results_df.to_csv(notebook_metrics_path, index=False)

for model_name, model in trained_models.items():
    joblib.dump(model, MODELS_DIR / f"notebook_{model_name}_best.joblib")

trained_results_df = pd.DataFrame(model_results).sort_values(
    ["f1_spam", "accuracy"], ascending=[False, False]
)
best_trained_model_name = trained_results_df.iloc[0]["model"]
joblib.dump(trained_models[best_trained_model_name], MODELS_DIR / "notebook_best_model.joblib")

print("Saved benchmark to:", notebook_metrics_path)
print("Saved evaluation to:", notebook_eval_path)
print("Saved classification report to:", notebook_cls_path)
print("Saved error analysis to:", notebook_err_path)
print("Best trained model in this notebook run:", best_trained_model_name)


## 10. Đánh giá chi tiết mô hình SVM dùng để demo

Dù benchmark hiện tại có thể cho thấy Logistic Regression nhỉnh hơn nhẹ, đồ án vẫn dùng **Linear SVM** làm mô hình demo
vì bám sát hướng đề tài và thuận tiện cho việc giải thích token đóng góp.


In [ ]:
svm_model = trained_models["svm"]
y_pred = svm_model.predict(X_test)
y_score = get_scores(svm_model, X_test)

svm_metrics = compute_metrics(y_test, y_pred, y_score)
svm_metrics["model"] = "notebook_svm"
svm_eval_df = pd.DataFrame([svm_metrics])
svm_eval_df.to_csv(notebook_eval_path, index=False)

class_report_df = build_classification_report(y_test, y_pred)
class_report_df.to_csv(notebook_cls_path, index=False)

error_df = build_error_analysis(test_df.copy(), y_test, y_pred, y_score)
error_df.to_csv(notebook_err_path, index=False)

svm_eval_df[
    [
        "model",
        "precision_ham",
        "recall_ham",
        "f1_ham",
        "precision_spam",
        "recall_spam",
        "f1_spam",
        "macro_f1",
        "balanced_accuracy",
        "accuracy",
        "fp",
        "fn",
        "tp",
        "tn",
    ]
]


## 10.1 Classification report theo từng lớp

Bảng này giúp giải thích riêng hiệu năng trên lớp `ham` và lớp `spam`.


In [ ]:
class_report_df


## 10.2 Confusion matrix của SVM

Confusion matrix giúp nhìn nhanh số lượng email:
- dự đoán đúng ham
- dự đoán đúng spam
- đoán nhầm ham thành spam
- đoán nhầm spam thành ham


In [ ]:
confusion_df = pd.DataFrame(
    [[svm_metrics["tn"], svm_metrics["fp"]], [svm_metrics["fn"], svm_metrics["tp"]]],
    index=["true_ham", "true_spam"],
    columns=["pred_ham", "pred_spam"],
)
confusion_df


## 11. Giải thích kết quả benchmark

Cell này giúp bạn nói rõ:
- mô hình nào đứng đầu
- SVM đang ở đâu
- khoảng cách giữa SVM và mô hình tốt nhất lớn hay nhỏ


In [ ]:
best_row = results_df.iloc[0]
svm_row = results_df[results_df["model"] == "svm"].iloc[0]
nb_row = results_df[results_df["model"] == "nb"].iloc[0]

print("Best model:", best_row["model"])
print("Best F1_spam:", round(best_row["f1_spam"] * 100, 4), "%")
print("SVM F1_spam:", round(svm_row["f1_spam"] * 100, 4), "%")
print("Naive Bayes F1_spam:", round(nb_row["f1_spam"] * 100, 4), "%")
print(
    "Khoảng cách F1 giữa best model và SVM:",
    round((best_row["f1_spam"] - svm_row["f1_spam"]) * 100, 4),
    "điểm %",
)


## 12. Phân tích lỗi dự đoán

Đây là các email mà mô hình SVM đoán sai.  
Phần này rất hữu ích khi vấn đáp vì bạn có thể chỉ ra các trường hợp khó như:
- email quá ngắn
- email có ngôn ngữ pha trộn
- email marketing nhưng nội dung giống thông báo thật
- email có obfuscation hoặc chuỗi ký tự lạ


In [ ]:
error_df.head(10)


## 13. Token quan trọng nhất của Linear SVM

Đây là cell giúp giải thích vì sao mô hình dự đoán spam hoặc ham.  
Trọng số dương lớn thường nghiêng về `spam`, còn trọng số âm lớn thường nghiêng về `ham`.


In [ ]:
svm_vectorizer = svm_model.named_steps["tfidf"]
svm_clf = svm_model.named_steps["clf"]

feature_names = svm_vectorizer.get_feature_names_out()
coef = svm_clf.coef_.ravel()

top_spam_tokens = pd.DataFrame(
    {
        "token": feature_names,
        "weight": coef,
    }
).sort_values("weight", ascending=False).head(15).reset_index(drop=True)

top_ham_tokens = pd.DataFrame(
    {
        "token": feature_names,
        "weight": coef,
    }
).sort_values("weight", ascending=True).head(15).reset_index(drop=True)

print("Top token nghiêng về spam")
display(top_spam_tokens)

print("Top token nghiêng về ham")
display(top_ham_tokens)


## 14. Dự đoán thử một số câu mẫu

Cell cuối dùng mô hình SVM đã huấn luyện để kiểm tra vài nội dung mô phỏng thực tế.  
Bạn có thể thay các câu này bằng ví dụ muốn trình bày trong lúc vấn đáp.


In [ ]:
def predict_text(model, text: str) -> dict:
    pred = int(model.predict([text])[0])
    score = get_scores(model, [text])
    score_value = None
    if score is not None:
        score_value = float(score[0]) if hasattr(score, "__len__") else float(score)
    return {
        "text": text,
        "predicted_label": "spam" if pred == 1 else "ham",
        "score": None if score_value is None else round(score_value, 6),
    }


sample_texts = [
    "Em chào cô, hôm nay cô khỏe không ạ? Em gửi cô file báo cáo nhóm để cô xem giúp.",
    "Chúc mừng! Bạn đã trúng thưởng iPhone 16, vui lòng nhấn vào đường link để xác nhận ngay.",
    "Tài khoản ngân hàng của bạn sắp bị khóa. Vui lòng xác minh thông tin ngay để tránh gián đoạn dịch vụ.",
]

pd.DataFrame([predict_text(svm_model, text) for text in sample_texts])


## 15. Tổng kết notebook

Sau khi chạy lần lượt từ trên xuống, notebook đã đi hết luồng của đồ án:

- dữ liệu gốc
- parse email
- tiền xử lý
- biểu diễn TF-IDF
- chia train/test
- baseline
- huấn luyện mô hình
- benchmark
- đánh giá chi tiết SVM
- phân tích lỗi
- dự đoán mẫu

Khi cần mở giao diện demo, dùng terminal riêng:

```bash
streamlit run app.py
```
